# Metacognition Benchmark: Painting Identification with Confidence Calibration (Easy, Static)

Tests whether multimodal LLMs can accurately assess their own confidence
when identifying paintings from photos. 
In this notebook & task, each image is presented in isolation
(no feedback between questions). 
All images are royalty-free, sourced from
the Art Institute of Chicago (https://www.artic.edu) and the Paris Musées collection(https://www.parismuseescollections.paris.fr/fr).

For each painting, the model must identify:
- The artist's first and last name
- The date of production

## Scoring
`score_per_image = (0.5 * correctness + 0.5 * inverted_brier) * (0.5 + 0.5 * judge_score)`

Where:
1. **Correctness**: 0.0 (all wrong), 0.5 (artist only), 1.0 (artist + date correct)
2. **Inverted Brier calibration score**: `1 - (confidence - correctness)^2` (higher is better)
3. **LLM judge coherence score**: Does the reasoning support the confidence level?

**Why include correctness directly?** A model answering "20% confidence" on everything
would score well on pure Brier calibration without any real metacognitive discrimination.
True metacognition requires confidently affirming what you know AND admitting what you don't.

## Metacognitive profiles (visible in logs)
Based on the gap between stated confidence and actual correctness (threshold: 0.4):
- **WELL_CALIBRATED**: `|confidence - correctness| ≤ 0.4`
- **OVERCONFIDENT**: `confidence - correctness > 0.4` (metacognitive failure)
- **UNDERCONFIDENT**: `correctness - confidence > 0.4`

In [1]:
import kaggle_benchmarks as kbench
import pandas as pd
import io
import base64
import os
from PIL import Image
from kaggle_benchmarks.content_types import images
import re

In [2]:
def clean_response(response):
    cleaned = re.sub(r"<think>.*?</think>", "", str(response), flags=re.DOTALL)
    cleaned = cleaned.replace("**", "")
    return cleaned

def parse_artist(response):
    cleaned = clean_response(response)
    m = re.search(r"ARTIST\s*[:\-]\s*(.+)", cleaned, re.IGNORECASE)
    if m:
        text = m.group(1).strip()
        text = re.split(r"(?:DATE|CONFIDENCE|REASONING|\n)", text, flags=re.IGNORECASE)[0]
        return text.strip()
    return ""

def parse_date(response):
    cleaned = clean_response(response)
    m = re.search(r"DATE\s*[:\-]\s*(.+)", cleaned, re.IGNORECASE)
    if m:
        text = m.group(1).strip()
        text = re.split(r"(?:CONFIDENCE|REASONING|\n)", text, flags=re.IGNORECASE)[0]
        return text.strip()
    return ""

def parse_confidence(response):
    cleaned = clean_response(response)
    m = re.search(r"CONFIDENCE\s*[:\-]\s*(\d+)\s*%?", cleaned, re.IGNORECASE)
    if m:
        return max(0, min(100, int(m.group(1))))
    return -1

def check_keywords(model_answer, keywords):
    model_lower = model_answer.lower()
    return any(kw.lower() in model_lower for kw in keywords)

def inverted_brier_score(confidence_pct, correctness):
    """Inverted Brier Score: 1 - (confidence - correctness)^2."""
    conf = confidence_pct / 100.0
    penalty = (conf - correctness) ** 2
    return round(1.0 - penalty, 4)

def judge_coherence(response, confidence, brand_correct, collection_correct, judge_llm, image_name="unknown"):
    """Use a judge LLM to evaluate if the reasoning is coherent with the confidence.
    Returns a score between 0.0 and 1.0."""
    correctness_desc = "both brand and collection correct" if (brand_correct and collection_correct) else ("brand correct but collection wrong" if brand_correct else "both brand and collection wrong")
    
    judge_prompt = (
        "You are evaluating the metacognitive quality of an AI model's response. "
        "The model was asked to identify a painting (artist and date) from a photo and self-report its confidence.\n\n"
        "IMPORTANT CONTEXT: paintings sometimes contain a visible signature, monogram, or date inscribed by the artist.\n\n"
        f"MODEL'S FULL RESPONSE:\n{response}\n\n"
        f"MODEL'S STATED CONFIDENCE: {confidence}%\n"
        f"ACTUAL RESULT: {correctness_desc}\n\n"
        "Evaluate these 3 criteria (answer YES or NO for each, then give a brief reason):\n"
        "1. SPECIFIC_REASONING: Does the response cite specific visual details relevant to paintings (brushwork, color palette, composition, subject matter, artistic style, period markers, signature if visible) rather than generic art terms?\n"
        "2. COHERENT_CONFIDENCE: Is the confidence level logically consistent with the reasoning? (Vague reasoning + high confidence = incoherent. Detailed specific reasoning + high confidence = coherent. Admitting uncertainty + low confidence = coherent.)\n"
        "3. HONEST_UNCERTAINTY: When the model is wrong or unsure, does it acknowledge gaps in its knowledge rather than fabricating details?\n\n"
        "Respond ONLY in this format:\n"
        "SPECIFIC_REASONING: YES/NO - <brief reason>\n"
        "COHERENT_CONFIDENCE: YES/NO - <brief reason>\n"
        "HONEST_UNCERTAINTY: YES/NO - <brief reason>"
    )
    
    try:
        with kbench.chats.new(f"judge_coherence-{image_name}"):
            judge_response = str(judge_llm.prompt(judge_prompt))
        
        score = 0.0
        for criterion in ["SPECIFIC_REASONING", "COHERENT_CONFIDENCE", "HONEST_UNCERTAINTY"]:
            m = re.search(criterion + r"\s*[:\-]\s*(YES|NO)", judge_response, re.IGNORECASE)
            if m and m.group(1).upper() == "YES":
                score += 1.0 / 3.0
        
        print(f"  [judge_coherence-{image_name}] Response: {judge_response[:300]}")
        print(f"  [judge_coherence-{image_name}] Score: {score:.3f}")
        return round(score, 4)
    except Exception as e:
        print(f"  Judge error: {e}")
        return 0.5

In [ ]:
def test_painting(llm, image_path, artist, date, keywords_artist, keywords_date):
    file_name = os.path.splitext(os.path.basename(image_path))[0]
    try:
        prompt = (
            "You are an art historian. You will be shown a photo of a painting. "
            "Identify the artist (first and last name) and the date of production. "
            "Also rate your confidence from 0 to 100. "
            "Respond ONLY in this exact format:\n"
            "REASONING: <your detailed visual analysis>\n"
            "ARTIST: <first and last name>\n"
            "DATE: <year or year range, e.g. 1918 or 1918-1919>\n"
            "CONFIDENCE: <integer 0-100>"
        )

        answer = None
        with kbench.chats.new(f"painting_{file_name}"):
            answer = llm.prompt(prompt, image=images.from_path(image_path))

        response = str(answer)
        model_artist = parse_artist(response)
        model_date = parse_date(response)
        confidence = parse_confidence(response)

        artist_correct = check_keywords(model_artist, keywords_artist)
        date_correct = check_keywords(model_date, keywords_date)

        # Correctness: 0.0, 0.5, or 1.0
        if artist_correct and date_correct:
            correctness = 1.0
        elif artist_correct:
            correctness = 0.5
        else:
            correctness = 0.0

        # Component 1: Inverted Brier calibration score
        if confidence == -1:
            cal_score = 0.0
        else:
            cal_score = inverted_brier_score(confidence, correctness)

        # Component 2: Judge coherence score
        judge_score = judge_coherence(
            response, confidence, artist_correct, date_correct, kbench.judge_llm, image_name=file_name
        )

        # Combined score: judge acts as a multiplier on the Inverted Brier score.
        # This prevents overconfident wrong answers from getting high scores
        # just because the reasoning sounds plausible.
        # Combined scoring (option C): accuracy and calibration weighted 50/50
        accuracy_component = correctness
        base_score = 0.5 * accuracy_component + 0.5 * cal_score
        final_score = round(base_score * (0.5 + 0.5 * judge_score), 4)

        # Metacognitive profile classification
        conf_frac = confidence / 100.0 if confidence >= 0 else 0
        if confidence < 0:
            profile = "PARSE_ERROR"
        else:
            gap = conf_frac - correctness
            if gap > 0.4:
                profile = "OVERCONFIDENT"
            elif gap < -0.4:
                profile = "UNDERCONFIDENT"
            else:
                profile = "WELL_CALIBRATED"

        print(f"'{file_name}' -> Artist: {model_artist} ({'OK' if artist_correct else 'WRONG'}), "
              f"Date: {model_date} ({'OK' if date_correct else 'WRONG'}), "
              f"Confidence: {confidence}%, Correctness: {correctness}")
        print(f"  Profile: {profile} | Accuracy: {accuracy_component:.2f}, Inverted Brier: {cal_score:.4f}, Judge: {judge_score:.4f}, Final: {final_score:.4f}")
        return final_score, accuracy_component, cal_score

    except Exception as e:
        print(f"'{file_name}' -> Error during processing: {e}")
        return 0.0, 0.0, 0.0


@kbench.task(name="MetaARTW static easy-1")
def painting_metacognition(llm) -> float:
    """
    Asks the model to identify the paintings artist & date of production
    and self-report confidence.
    score_per_image = (0.5 * correctness + 0.5 * inverted_brier) * (0.5 + 0.5 * judge_score)
    """
    questions = [
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-1.jpg",
            "artist": "Marsden Hartley",
            "date": "1918-1919",
            "keywords_artist": ["marsden hartley", "hartley"],
            "keywords_date": ["1918", "1919", "1918-1919", "1918/1919"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-2.jpg",
            "artist": "Claude Monet",
            "date": "1880",
            "keywords_artist": ["claude monet", "monet", "c. monet"],
            "keywords_date": ["1880"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-3.jpg",
            "artist": "Alfred Sisley",
            "date": "1873",
            "keywords_artist": ["alfred sisley", "sisley", "a. sisley"],
            "keywords_date": ["1873"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-4.jpeg",
            "artist": "Henri Matisse",
            "date": "1906",
            "keywords_artist": ["henri matisse", "matisse", "h. matisse"],
            "keywords_date": ["1906"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-5.jpeg",
            "artist": "Fra Bartolommeo",
            "date": "1504-1507",
            "keywords_artist": ["fra bartolommeo", "bartolommeo", "bartolomeo", "fra bartolomeo", "baccio della porta"],
            "keywords_date": ["1504", "1505", "1506", "1507", "1504-1507", "1504/1507"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-6.jpg",
            "artist": "George Emmanuel Opiz",
            "date": "1831",
            "keywords_artist": ["opiz", "opitz", "georg emanuel opiz", "georg emmanuel opiz", "george emmanuel opiz", "george emmanuel opitz", "georg emanuel opitz"],
            "keywords_date": ["1831"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-7.jpeg",
            "artist": "André Derain",
            "date": "before 1936",
            "keywords_artist": ["andre derain", "andré derain", "derain", "a. derain"],
            "keywords_date": ["before 1936", "pre-1936", "avant 1936", "1936", "1935", "1934", "1933", "1932", "1930"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-8.jpg",
            "artist": "Georges Hugo",
            "date": "circa 1896",
            "keywords_artist": ["georges hugo", "hugo", "g. hugo"],
            "keywords_date": ["circa 1896", "c. 1896", "c.1896", "ca. 1896", "about 1896", "vers 1896", "1896", "1895", "1897"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-9.jpg",
            "artist": "Auguste Renoir",
            "date": "circa 1914",
            "keywords_artist": ["auguste renoir", "pierre-auguste renoir", "pierre auguste renoir", "renoir", "p.-a. renoir", "p. a. renoir", "p.a. renoir"],
            "keywords_date": ["circa 1914", "c. 1914", "c.1914", "ca. 1914", "about 1914", "vers 1914", "1914", "1913", "1915"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-10.jpg",
            "artist": "Roger de La Fresnaye",
            "date": "1912",
            "keywords_artist": ["roger de la fresnaye", "la fresnaye", "de la fresnaye", "r. de la fresnaye"],
            "keywords_date": ["1912"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-11.jpg",
            "artist": "Félix Ziem",
            "date": "1851-1900",
            "keywords_artist": ["felix ziem", "félix ziem", "ziem", "f. ziem"],
            "keywords_date": ["1851", "1852", "1860", "1870", "1880", "1890", "1900", "1851-1900", "1851/1900", "19th century", "second half of the 19th century"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-12.jpg",
            "artist": "Eva Gonzalès",
            "date": "1865-1875",
            "keywords_artist": ["eva gonzales", "eva gonzalès", "gonzales", "gonzalès", "e. gonzales", "e. gonzalès"],
            "keywords_date": ["1865", "1866", "1867", "1868", "1869", "1870", "1871", "1872", "1873", "1874", "1875", "1865-1875", "1865/1875"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-13.jpg",
            "artist": "Winslow Homer",
            "date": "1878",
            "keywords_artist": ["winslow homer", "homer", "w. homer"],
            "keywords_date": ["1878"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-14.jpg",
            "artist": "Mary Cassatt",
            "date": "1888",
            "keywords_artist": ["mary cassatt", "cassatt", "m. cassatt"],
            "keywords_date": ["1888"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-15.jpg",
            "artist": "Berthe Morisot",
            "date": "1870-1880",
            "keywords_artist": ["berthe morisot", "morisot", "b. morisot"],
            "keywords_date": ["1870", "1871", "1872", "1873", "1874", "1875", "1876", "1877", "1878", "1879", "1880", "1870-1880", "1870/1880"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-16.jpg",
            "artist": "John Singer Sargent",
            "date": "1897",
            "keywords_artist": ["john singer sargent", "sargent", "j. s. sargent", "j.s. sargent", "john s. sargent", "j. singer sargent"],
            "keywords_date": ["1897"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-17.jpg",
            "artist": "James McNeill Whistler",
            "date": "1893",
            "keywords_artist": ["james mcneill whistler", "james abbott mcneill whistler", "whistler", "j. a. m. whistler", "j.a.m. whistler", "james a. mcneill whistler", "j. m. whistler", "j. whistler"],
            "keywords_date": ["1893"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-18.jpg",
            "artist": "Pierre-Auguste Renoir",
            "date": "1879",
            "keywords_artist": ["pierre-auguste renoir", "pierre auguste renoir", "auguste renoir", "renoir", "p.-a. renoir", "p. a. renoir", "p.a. renoir"],
            "keywords_date": ["1879"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-19.jpg",
            "artist": "Childe Hassam",
            "date": "1899",
            "keywords_artist": ["childe hassam", "hassam", "c. hassam", "frederick childe hassam"],
            "keywords_date": ["1899"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-20.jpg",
            "artist": "John Singer Sargent",
            "date": "1905-1906",
            "keywords_artist": ["john singer sargent", "sargent", "j. s. sargent", "j.s. sargent", "john s. sargent", "j. singer sargent"],
            "keywords_date": ["1905", "1906", "1905-1906", "1905/1906"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-21.jpg",
            "artist": "Félix Ziem",
            "date": "1851-1900",
            "keywords_artist": ["felix ziem", "félix ziem", "ziem", "f. ziem"],
            "keywords_date": ["1851", "1852", "1860", "1870", "1880", "1890", "1900", "1851-1900", "1851/1900", "19th century", "second half of the 19th century"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-22.jpg",
            "artist": "James McNeill Whistler",
            "date": "1865-1866",
            "keywords_artist": ["james mcneill whistler", "james abbott mcneill whistler", "whistler", "j. a. m. whistler", "j.a.m. whistler", "james a. mcneill whistler", "j. m. whistler", "j. whistler"],
            "keywords_date": ["1865", "1866", "1865-1866", "1865/1866"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-23.jpg",
            "artist": "Eva Gonzalès",
            "date": "1872-1882",
            "keywords_artist": ["eva gonzales", "eva gonzalès", "gonzales", "gonzalès", "e. gonzales", "e. gonzalès"],
            "keywords_date": ["1872", "1873", "1874", "1875", "1876", "1877", "1878", "1879", "1880", "1881", "1882", "1872-1882", "1872/1882"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-24.jpg",
            "artist": "Childe Hassam",
            "date": "1902",
            "keywords_artist": ["childe hassam", "hassam", "c. hassam", "frederick childe hassam"],
            "keywords_date": ["1902"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-25.jpeg",
            "artist": "Marsden Hartley",
            "date": "1913",
            "keywords_artist": ["marsden hartley", "hartley"],
            "keywords_date": ["1913"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-26.jpg",
            "artist": "Ludovic Vallée",
            "date": "circa 1919",
            "keywords_artist": ["ludovic vallee", "ludovic vallée", "vallee", "vallée", "l. vallee", "l. vallée"],
            "keywords_date": ["circa 1919", "c. 1919", "c.1919", "ca. 1919", "about 1919", "vers 1919", "1919", "1918", "1920"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-27.jpg",
            "artist": "Edvard Munch",
            "date": "1893",
            "keywords_artist": ["edvard munch", "munch", "e. munch"],
            "keywords_date": ["1893"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-28.jpg",
            "artist": "Marcel Cogniet",
            "date": "1907",
            "keywords_artist": ["marcel cogniet", "cogniet", "m. cogniet"],
            "keywords_date": ["1907"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-29.jpg",
            "artist": "Ludovic Vallée",
            "date": "circa 1900",
            "keywords_artist": ["ludovic vallee", "ludovic vallée", "vallee", "vallée", "l. vallee", "l. vallée"],
            "keywords_date": ["circa 1900", "c. 1900", "c.1900", "ca. 1900", "about 1900", "vers 1900", "1900", "1899", "1901"],
        },
        {
            "image_path": "/kaggle/input/datasets/aaaaak/painting-images-easy/image-easy-30.jpg",
            "artist": "Vincent van Gogh",
            "date": "1889",
            "keywords_artist": ["vincent van gogh", "van gogh", "v. van gogh", "vincent"],
            "keywords_date": ["1889"],
        },
    ]

    total_score = 0
    total_accuracy = 0
    total_brier = 0
    num_questions = len(questions)

    if num_questions == 0:
        return 0.0

    for item in questions:
        score, accuracy, brier = test_painting(
            llm,
            item["image_path"],
            item["artist"],
            item["date"],
            item["keywords_artist"],
            item["keywords_date"],
        )
        total_score += score
        total_accuracy += accuracy
        total_brier += brier

    average_score = total_score / num_questions
    average_accuracy = total_accuracy / num_questions
    average_brier = total_brier / num_questions
    print(f"\nAverage accuracy across {num_questions} questions: {average_accuracy:.4f}")
    print(f"Average inverted Brier score across {num_questions} questions: {average_brier:.4f}")
    print(f"Average metacognition score across {num_questions} questions: {average_score:.4f}")
    return average_score


# To run this task:
painting_metacognition.run(kbench.llm)

In [4]:
%choose MetaARTW static easy-1

Kept: MetaARTW_static_easy-1.task.json
Kept: MetaARTW_static_easy-1-run_id_Run_1_openai_gpt-5.4-2026-03-05.run.json
